In [8]:
from pathlib import Path
import sys

# Identifier la racine du projet
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Permettre l'importation depuis src
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [24]:
import pandas as pd
from src.config import SILVER_PARQUET_PATH

# Lire le fichier Silver
df_orders = pd.read_parquet(
    SILVER_PARQUET_PATH / "orders.parquet"
)

# Afficher les 5 premières lignes
display(df_orders.head())
print("\nTypes des colonnes :")
print(df_orders.dtypes)

print("\nValeurs manquantes :")
print(df_orders.isna().sum())


    # Vérifier la longueur maximale des colonnes texte
text_columns = [
        "order_id",
        "customer_id",
        "order_status"
    ]

for col in text_columns:
        longueur_max = df_orders[col].str.len().max()
        print(f"{col} : {longueur_max} caractères")


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,anomalie_livraison
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False



Types des colonnes :
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
anomalie_livraison                         bool
dtype: object

Valeurs manquantes :
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
anomalie_livraison                  0
dtype: int64
order_id : 32 caractères
customer_id : 32 caractères
order_status : 11 caractères


In [36]:
# Convertir les dates de livraison
df_orders = pd.read_parquet(
    SILVER_PARQUET_PATH / "orders.parquet"
)
date_livraison = pd.to_datetime(
    df_orders["order_delivered_customer_date"]
)

date_estimee = pd.to_datetime(
    df_orders["order_estimated_delivery_date"]
)

# Sélectionner les commandes ayant une date de livraison
commandes_livrees = df_orders[
    date_livraison.notna()
].copy()

# Identifier les retards
commandes_livrees["en_retard"] = (
    date_livraison.loc[commandes_livrees.index].dt.normalize()
    > date_estimee.loc[commandes_livrees.index].dt.normalize()
)

In [37]:
# Convertir la date estimée
date_estimee = pd.to_datetime(
    df_orders["order_estimated_delivery_date"]
)

# Conserver les commandes avec une date réelle de livraison
commandes_livrees = df_orders[
    date_livraison.notna()
].copy()

# Identifier les retards en comparant les dates calendaires
commandes_livrees["en_retard"] = (
    date_livraison.loc[commandes_livrees.index].dt.normalize()
    > date_estimee.loc[commandes_livrees.index].dt.normalize()
)

# Afficher le nombre de commandes en retard
print("Commandes en retard :", commandes_livrees["en_retard"].sum())

Commandes en retard : 6535


In [38]:
# Vérifier les identifiants de commandes dupliqués
nb_doublons = df_orders["order_id"].duplicated().sum()

print("Nombre de commandes dupliquées :", nb_doublons)

Nombre de commandes dupliquées : 0


In [39]:
# Commandes livrées sans date de livraison au client
anomalies_livraison = df_orders[
    (df_orders["order_status"] == "delivered")
    & (df_orders["order_delivered_customer_date"].isnull())
]

# Afficher les commandes concernées
anomalies_livraison

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,anomalie_livraison
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18,True
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16,True
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30,True
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30,True
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24,True
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23,True
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26,True
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19,True


In [40]:
# Sélectionner les commandes sans date de livraison
commandes_sans_livraison = df_orders[
    df_orders["order_delivered_customer_date"].isnull()
]

# Compter les commandes par statut
commandes_sans_livraison["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [41]:
df_orders["order_status"].value_counts()


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [42]:
# Pourcentage de valeurs manquantes
(df_orders.isnull().sum() / len(df_orders) * 100).round(2)

order_id                         0.00
customer_id                      0.00
order_status                     0.00
order_purchase_timestamp         0.00
order_approved_at                0.16
order_delivered_carrier_date     1.79
order_delivered_customer_date    2.98
order_estimated_delivery_date    0.00
anomalie_livraison               0.00
dtype: float64

In [43]:
# Compter les valeurs manquantes par colonne
df_orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
anomalie_livraison                  0
dtype: int64

In [44]:
# Afficher les dimensions du DataFrame
print("Nombre de lignes :", df_orders.shape[0])
print("Nombre de colonnes :", df_orders.shape[1])

# Afficher les colonnes et leurs types
df_orders.info()

Nombre de lignes : 99441
Nombre de colonnes : 9
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
 8   anomalie_livraison             99441 non-null  bool          
dtypes: bool(1), datetime64[ns](5), object(3)
memory usage: 6.2+ MB


In [45]:
# Convertir temporairement les dates pour l'analyse
date_achat = pd.to_datetime(df_orders["order_purchase_timestamp"])
date_livraison = pd.to_datetime(df_orders["order_delivered_customer_date"])

# Identifier les livraisons antérieures aux achats
anomalies_dates = df_orders[
    date_livraison < date_achat
]

print("Nombre d'anomalies chronologiques :", len(anomalies_dates))

Nombre d'anomalies chronologiques : 0


In [46]:
from pathlib import Path
import pandas as pd

# Chemin vers le dossier Bronze
bronze_path = Path.cwd().parent / "data" / "bronze"

# Charger les commandes
df_orders = pd.read_csv(bronze_path / "olist_orders_dataset.csv")

# Afficher les 5 premières lignes
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
